# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umerkang66/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook establishes the transparent, rule-based baseline queue for **Lane 2: Refresh / Content Opportunity Scoring** (locked and confirmed).

A model without a baseline is a number without a meaning. Before fitting complex supervised models in Week 5, we encode editorial intuition into a transparent, hand-written rule. We test the underlying empirical signals first, attach explicit reason codes and action labels, evaluate ranked precision against the portfolio base rate, and examine our top picks with a skeptic's eye.

> **Skills loaded:** `building-baselines` + `flyrank/flyrank-data`

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Plain Words Definition (Three Sentences)
A content item is flagged for editorial refresh if it has not been updated in at least 180 days and still maintains active search exposure (at least 500 impressions over the trailing 90 days). Eligible pages are ranked by their 90-day search volume so editorial teams prioritize high-exposure decaying assets first. Content falling below these thresholds is labeled for ongoing passive monitoring rather than immediate manual intervention.

### Action Label & Reason Codes
- **Action Label:** `refresh` (for flagged opportunities) or `monitor` (for unflagged assets).
- **Reason Code:**
  - `stale_visible_page`: Content exceeds the 180-day staleness threshold and has at least 500 trailing-90-day search impressions.
  - `below_threshold`: Content does not satisfy the joint staleness and search visibility gates.

### Pre-Rule Signal Checks (Empirical Signal Audit)
Before locking this rule, we audit two underlying signals against observed decay (`is_declining_label == 1`), printing full bucket tables with row counts ($n$), declining counts, observed decline rates, and lift over the portfolio base rate ($54.2\%$):

1. **Signal 1 (Flag-Linked): Staleness (`days_since_last_update`)**
   - *Intuition behind FlyRank refresh flags:* Content left unmaintained loses relevance, search intent shifts, and organic visibility degrades.
   - *Empirical Finding:* Between 0 and 180 days, decline rates rise monotonically from $51.1\%$ (0–30d) to $61.1\%$ (91–180d, $n=9,171$). However, at 181–365 days, the decline rate paradoxically drops to $46.7\%$ ($n=169$, below the portfolio average). This reveals a survivor bias: pages surviving $>180$ days without updates often represent stable evergreen assets or client-specific batch logging gaps.
   - *One-Word Verdict:* **MIXED** (Peak decline occurs at 91–180 days; assuming strict monotonic decay past 180 days is empirically flawed).

2. **Signal 2 (Demand-Linked): Search Visibility (`impressions_90d`)**
   - *Intuition behind visibility thresholds & quick-win flags:* Pages with high search impressions face competitive churn and algorithm exposure; zero/low-demand pages are stagnant.
   - *Empirical Finding:* Pages with active impressions ($100$ to $5,000+$) exhibit decline rates between $54.7\%$ and $63.4\%$, compared to only $38.9\%$ for low-visibility assets ($<100$ impressions, $n=8,006$).
   - *One-Word Verdict:* **CONFIRMED** (Search exposure is a prerequisite for traffic decay and business upside).

3. **Signal 3 (Bonus Flag-Linked): CTR on Page 1 (`avg_position <= 10`)**
   - *Intuition behind FlyRank CTR-fix flags:* Ranking on Page 1 with below-average CTR signals snippet/title decay, prompting search engine demotion.
   - *Empirical Finding:* For Page 1 assets, low CTR ($\le 0.05\%$) shows a elevated decline rate ($56.7\%\text{--}66.6\%$) vs. only $43.5\%$ for healthy CTR ($>1.00\%$).
   - *One-Word Verdict:* **CONFIRMED**.

In [1]:
# Signal Audit & Verification Code
import os
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve dataset path safely across local repo and Colab
DATA_PATHS = [
    Path('data/raw/content_refresh_anonymized.csv'),
    Path('../../data/raw/content_refresh_anonymized.csv'),
    Path('../data/raw/content_refresh_anonymized.csv')
]
data_file = next((p for p in DATA_PATHS if p.exists()), None)
if data_file is None:
    raise FileNotFoundError("Starter dataset data/raw/content_refresh_anonymized.csv not found.")

df = pd.read_csv(data_file)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base_rate = float(df["is_declining_label"].mean())

print(f"Total Content Items: {len(df):,} | Unique Clients: {df['client_id'].nunique()}")
print(f"Portfolio Base Decline Rate: {base_rate:.4f} ({base_rate*100:.2f}%)\n")

# --- Signal 1: Staleness (days_since_last_update) ---
bins_stale = [-1, 30, 60, 90, 180, 365, 1000]
labels_stale = ['0-30d', '31-60d', '61-90d', '91-180d', '181-365d', '365d+']
df['stale_bucket'] = pd.cut(df['days_since_last_update'], bins=bins_stale, labels=labels_stale)
stale_table = df.groupby('stale_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_count=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()
stale_table['lift_vs_base'] = stale_table['decline_rate'] / base_rate

print("=== SIGNAL 1 (FLAG-LINKED): STALENESS (days_since_last_update) ===")
print(stale_table.to_string(index=False, formatters={
    'n': '{:,}'.format,
    'declining_count': '{:,}'.format,
    'decline_rate': '{:.3f}'.format,
    'lift_vs_base': '{:.2f}x'.format
}))
print("VERDICT: MIXED -- Risk peaks at 91-180d (61.1%), but dips at 180d+ (46.7%) due to evergreen survivor bias.\n")

# --- Signal 2: Search Visibility (impressions_90d) ---
bins_imp = [-1, 100, 500, 1000, 5000, 10_000_000]
labels_imp = ['<100', '100-499', '500-999', '1k-5k', '5k+']
df['imp_bucket'] = pd.cut(df['impressions_90d'], bins=bins_imp, labels=labels_imp)
imp_table = df.groupby('imp_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_count=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()
imp_table['lift_vs_base'] = imp_table['decline_rate'] / base_rate

print("=== SIGNAL 2 (DEMAND-LINKED): SEARCH VISIBILITY (impressions_90d) ===")
print(imp_table.to_string(index=False, formatters={
    'n': '{:,}'.format,
    'declining_count': '{:,}'.format,
    'decline_rate': '{:.3f}'.format,
    'lift_vs_base': '{:.2f}x'.format
}))
print("VERDICT: CONFIRMED -- Active impression assets suffer 60-63% decline vs 38.9% for low-demand pages.\n")

# --- Signal 3 (Bonus): CTR on Page 1 (avg_position <= 10) ---
p1 = df[(df['avg_position'] > 0) & (df['avg_position'] <= 10)].copy()
bins_ctr = [-0.001, 0.05, 0.10, 0.50, 1.00, 100.0]
labels_ctr = ['<=0.05%', '0.05-0.10%', '0.10-0.50%', '0.50-1.00%', '>1.00%']
p1['ctr_bucket'] = pd.cut(p1['ctr'], bins=bins_ctr, labels=labels_ctr)
ctr_table = p1.groupby('ctr_bucket', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_count=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean')
).reset_index()
ctr_table['lift_vs_base'] = ctr_table['decline_rate'] / base_rate

print("=== SIGNAL 3 (BONUS FLAG-LINKED): CTR ON PAGE 1 (avg_position <= 10) ===")
print(ctr_table.to_string(index=False, formatters={
    'n': '{:,}'.format,
    'declining_count': '{:,}'.format,
    'decline_rate': '{:.3f}'.format,
    'lift_vs_base': '{:.2f}x'.format
}))
print("VERDICT: CONFIRMED -- Sub-0.10% CTR on Page 1 declines at 56.7%-66.6% vs 43.5% for CTR > 1.00%.")

Total Content Items: 30,000 | Unique Clients: 32
Portfolio Base Decline Rate: 0.5421 (54.21%)

=== SIGNAL 1 (FLAG-LINKED): STALENESS (days_since_last_update) ===
stale_bucket      n declining_count decline_rate lift_vs_base
       0-30d 20,480          10,473        0.511        0.94x
      31-60d    128              75        0.586        1.08x
      61-90d     47              28        0.596        1.10x
     91-180d  9,171           5,604        0.611        1.13x
    181-365d    169              79        0.467        0.86x
       365d+      5               3        0.600        1.11x
VERDICT: MIXED -- Risk peaks at 91-180d (61.1%), but dips at 180d+ (46.7%) due to evergreen survivor bias.

=== SIGNAL 2 (DEMAND-LINKED): SEARCH VISIBILITY (impressions_90d) ===
imp_bucket     n declining_count decline_rate lift_vs_base
      <100 8,006           3,116        0.389        0.72x
   100-499 5,279           3,190        0.604        1.11x
   500-999 3,206           1,925        0.600    

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Encoding the Transparent Rule
Following the live session formulation and `building-baselines` skill, we code a transparent score with zero fitted weights:

$$\text{stale} = \mathbb{I}(\text{days\_since\_last\_update} \ge 180)$$
$$\text{visible} = \mathbb{I}(\text{impressions\_90d} \ge 500)$$
$$\text{baseline\_action\_score} = \text{stale} \times \text{visible} \times \text{impressions\_90d}$$

### Ranked Precision@K vs. Portfolio Base Rate
To evaluate ranking efficacy honestly, we measure **Precision@K**: of the top $K$ items surfaced by the score, what fraction are actually declining in organic search? We compare this directly to random selection (portfolio base rate $= 54.2\%$):
- **Precision@10:** $1.000$ ($10/10$ decaying, **$1.84\times$ lift** over base rate)
- **Precision@20:** $0.900$ ($18/20$ decaying, **$1.66\times$ lift** over base rate)
- **Precision@50:** $0.600$ ($30/50$ decaying, **$1.11\times$ lift** over base rate)
- **Precision@100:** $0.490$ ($49/100$ decaying, **$0.90\times$ lift** — drops below base rate as unflagged ties enter)

The ranked queue is written to `work/outputs/baseline_action_score.csv`, and run metadata receipts are saved to `work/outputs/baseline_metrics.json`.

In [2]:
# Compute deterministic baseline score, rank queue, export CSV & JSON receipts
import json

# Construct deterministic rule components
stale_cond = (df['days_since_last_update'] >= 180).astype(int)
visible_cond = (df['impressions_90d'] >= 500).astype(int)

df['baseline_action_score'] = stale_cond * visible_cond * df['impressions_90d']
df['reason_code'] = np.where(df['baseline_action_score'] > 0, 'stale_visible_page', 'below_threshold')
df['action_label'] = np.where(df['baseline_action_score'] > 0, 'refresh', 'monitor')

# Sort ranked queue: primary by baseline_action_score, secondary by impressions_90d, stable tie-break by content_id
ranked_queue = df.sort_values(
    by=['baseline_action_score', 'impressions_90d', 'content_id'],
    ascending=[False, False, True]
).reset_index(drop=True)
ranked_queue['rank'] = ranked_queue.index + 1

# Export ranked queue to work/outputs/baseline_action_score.csv
OUTPUT_DIR = Path('work/outputs') if Path('work').exists() else Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / 'baseline_action_score.csv'
OUTPUT_METRICS = OUTPUT_DIR / 'baseline_metrics.json'

export_cols = [
    'rank',
    'content_id',
    'client_id',
    'baseline_action_score',
    'reason_code',
    'action_label',
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'content_age_days',
    'word_count',
    'trend_direction',
    'is_declining_label'
]
ranked_queue[export_cols].to_csv(OUTPUT_CSV, index=False)
print(f"Successfully wrote ranked queue to: work/outputs/baseline_action_score.csv ({len(ranked_queue):,} rows)")

# Precision@K evaluation function
def precision_at_k(queue, k):
    return float(queue.head(k)['is_declining_label'].mean())

p10 = precision_at_k(ranked_queue, 10)
p20 = precision_at_k(ranked_queue, 20)
p50 = precision_at_k(ranked_queue, 50)
p100 = precision_at_k(ranked_queue, 100)

print("\n=== RANKED BASELINE EVALUATION (PRECISION@K) ===")
print(f"  Random Floor (Portfolio Base Rate): {base_rate:.4f}")
print(f"  Precision@10:  {p10:.4f}  (Lift: {p10/base_rate:.2f}x | {int(p10*10)}/10 correct)")
print(f"  Precision@20:  {p20:.4f}  (Lift: {p20/base_rate:.2f}x | {int(p20*20)}/20 correct)")
print(f"  Precision@50:  {p50:.4f}  (Lift: {p50/base_rate:.2f}x | {int(p50*50)}/50 correct)")
print(f"  Precision@100: {p100:.4f}  (Lift: {p100/base_rate:.2f}x | {int(p100*100)}/100 correct)")

# Save metric receipts to work/outputs/baseline_metrics.json
metrics_payload = {
    "task": "ML-07 Baseline Action Score",
    "lane": "Lane 2 — Refresh / Content Opportunity Scoring",
    "dataset_rows": int(len(df)),
    "base_decline_rate": round(base_rate, 4),
    "rule": {
        "name": "Stale Visible Refresh Rule",
        "formula": "(days_since_last_update >= 180) * (impressions_90d >= 500) * impressions_90d",
        "reason_code": "stale_visible_page",
        "action_label": "refresh",
        "unflagged_action_label": "monitor"
    },
    "signal_verdicts": {
        "days_since_last_update": "MIXED",
        "impressions_90d": "CONFIRMED",
        "ctr_on_page_1": "CONFIRMED"
    },
    "precision_metrics": {
        "precision_at_10": round(p10, 4),
        "precision_at_20": round(p20, 4),
        "precision_at_50": round(p50, 4),
        "precision_at_100": round(p100, 4),
        "lift_at_10": round(p10 / base_rate, 2),
        "lift_at_20": round(p20 / base_rate, 2)
    },
    "flagged_rows_count": int((df['baseline_action_score'] > 0).sum()),
    "ranked_csv": "work/outputs/baseline_action_score.csv"
}

with open(OUTPUT_METRICS, 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)
print(f"\nWrote run receipts JSON to: work/outputs/baseline_metrics.json")

Successfully wrote ranked queue to: work/outputs/baseline_action_score.csv (30,000 rows)

=== RANKED BASELINE EVALUATION (PRECISION@K) ===
  Random Floor (Portfolio Base Rate): 0.5421
  Precision@10:  1.0000  (Lift: 1.84x | 10/10 correct)
  Precision@20:  0.9000  (Lift: 1.66x | 18/20 correct)
  Precision@50:  0.6000  (Lift: 1.11x | 30/50 correct)
  Precision@100: 0.4900  (Lift: 0.90x | 49/100 correct)

Wrote run receipts JSON to: work/outputs/baseline_metrics.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top of the queue is where rule logic reveals both its practical utility and its blind spots. Below is the line-by-line review of all top 20 assets:

### Line-by-Line Skeptic's Audit

1. **Rank 1 (`content_cf56e2e2e282`)** | Score: 61,678 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
   - *Why it's there:* Top impression asset in the staleness window (61.7k impressions, 94 clicks, pos 19.7, 194 days stale).
   - *What would make it wrong:* At 5,125 words, this piece is already exhaustive; the decline may reflect SERP feature displacement (AI Overviews, video carousels) rather than outdated copy.

2. **Rank 2 (`content_7368877ea310`)** | Score: 59,472 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
   - *Why it's there:* Heavy impression exposure (59.5k impressions, 77 clicks, pos 24.8, 194 days stale).
   - *What would make it wrong:* Position 24.8 places this on Page 3. Organic search traffic rarely ventures past Page 1; rewriting without domain authority or backlink signals will yield negligible click recovery.

3. **Rank 3 (`content_1bfaa38ff26c`)** | Score: 25,715 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
   - *Why it's there:* 25.7k impressions, pos 22.2, 194 days stale, 3,861 words.
   - *What would make it wrong:* Sitting on Page 3; traffic decline could be seasonal demand contraction rather than topical obsolescence.

4. **Rank 4 (`content_0a91db491d14`)** | Score: 13,299 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
   - *Why it's there:* Striking-distance asset on the cusp of Page 1 (pos 10.5, 13.3k impressions, 65 clicks, 193 days stale).
   - *What would make it wrong:* Already maintains a solid 0.49% CTR. If search intent transitioned from informational to transactional, a simple content refresh will fail to satisfy user search journeys.

5. **Rank 5 (`content_5feee3994adb`)** | Score: 7,812 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Low
   - *Why it's there:* 7.8k impressions, 194 days stale, transactional intent.
   - *What would make it wrong:* Deep Page 4 ranking (pos 39.0) with only 1 click in 90 days. Google does not view this domain as a qualified transactional merchant; editorial updates cannot overcome entity mismatch.

6. **Rank 6 (`content_c2d929d83eaa`)** | Score: 7,558 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Medium
   - *Why it's there:* Striking distance (pos 17.9, 7.6k impressions, 15 clicks, 193 days stale).
   - *What would make it wrong:* Piece is 4,758 words long; expanding it further risks keyword dilution and reader fatigue.

7. **Rank 7 (`content_b16bd7307b39`)** | Score: 4,590 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Low
   - *Why it's there:* 4.6k impressions, 194 days stale, 4,329 words.
   - *What would make it wrong:* Zero clicks in 90 days at pos 31.0; page earns accidental impressions on broad non-converting queries.

8. **Rank 8 (`content_fe16a55cd13d`)** | Score: 4,556 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
   - *Why it's there:* Classic striking opportunity (pos 16.4, 4.6k impressions, 15 clicks, 194 days stale).
   - *What would make it wrong:* If rank loss is driven by technical performance (Core Web Vitals or mobile rendering), editorial updates won't reverse it.

9. **Rank 9 (`content_ecb6215e79fd`)** | Score: 4,429 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Medium
   - *Why it's there:* 4.4k impressions, 194 days stale, 4,486 words, pos 25.3.
   - *What would make it wrong:* Ranks on Page 3; traffic drop could reflect seasonal industry dips rather than content decay.

10. **Rank 10 (`content_928af3e22c80`)** | Score: 1,697 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Medium
    - *Why it's there:* Pos 15.8, 1.7k impressions, 193 days stale.
    - *What would make it wrong:* Low engagement velocity (only 2 clicks); competitors may have adopted rich media/calculators that static prose cannot rival.

11. **Rank 11 (`content_e3ff1b093148`)** | Score: 1,408 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
    - *Why it's there:* Page 1 rank (pos 7.8, 1.4k impressions, 183 days stale).
    - *What would make it wrong:* Ranks on Page 1 but CTR is weak (0.28%); the required action is a title/snippet rewrite (CTR fix), not a heavy content overhaul.

12. **Rank 12 (`content_bdbec75c1148`)** | Score: 1,316 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: False Positive (Wrong)
    - *Why it's there:* Satisfies 194 days staleness and 1.3k impressions.
    - *What would make it wrong:* **FALSE POSITIVE.** Its observed trend is `stable` (`is_declining_label == 0`). Assigning editorial hours here wastes resources on healthy content.

13. **Rank 13 (`content_7f116ae1f6f5`)** | Score: 954 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
    - *Why it's there:* Page 1 asset (pos 9.0, 954 impressions, 301 days stale, 1,335 words).
    - *What would make it wrong:* If users seek brief, direct answers, expanding word count could hurt user engagement.

14. **Rank 14 (`content_77d4d5930e5e`)** | Score: 828 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Low
    - *Why it's there:* Pos 18.6, 828 impressions, 194 days stale.
    - *What would make it wrong:* Low total demand (~9 impressions/day); revenue ceiling does not justify editorial hours.

15. **Rank 15 (`content_72496874f806`)** | Score: 821 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: High
    - *Why it's there:* Page 1 asset (pos 5.8, 821 impressions, 301 days stale).
    - *What would make it wrong:* Low CTR (0.24%) at position 5.8 indicates SERP presentation issues rather than body copy staleness.

16. **Rank 16 (`content_6226ee6adc91`)** | Score: 545 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Low
    - *Why it's there:* Pos 17.8, 545 impressions, 183 days stale.
    - *What would make it wrong:* Barely exceeds the 500 impression threshold; minimal traffic upside.

17. **Rank 17 (`content_074ba6ead17b`)** | Score: 533 | Action: `refresh` | Reason: `stale_visible_page` | Confidence: Low
    - *Why it's there:* 533 impressions, 183 days stale, 3,994 words.
    - *What would make it wrong:* Position 48.0 (Page 5) and 0 clicks. Refreshing will not rescue a page with no baseline ranking traction.

18. **Rank 18 (`content_5fe46e04994d`)** | Score: 0 | Action: `monitor` | Reason: `below_threshold` | Confidence: False Negative (Severe)
    - *Why it's there:* High impression volume (517,715 impressions, pos 4.2), actively decaying (`trend_direction == 'down'`), but scored 0 because `days_since_last_update` = 104 (< 180).
    - *What would make it wrong:* **CATASTROPHIC FALSE NEGATIVE.** The rule advises 'monitor' while a half-million impression Page 1 flagship asset collapses. Hard thresholds miss severe medium-staleness decay.

19. **Rank 19 (`content_aaef01a50def`)** | Score: 0 | Action: `monitor` | Reason: `below_threshold` | Confidence: Valid Monitor
    - *Why it's there:* 517k impressions, pos 5.4, updated 22 days ago, trend is `stable`.
    - *What would make it wrong:* Monitoring is correct here, but the rule gives it the exact same score (0) as Rank 18, demonstrating complete lack of nuance.

20. **Rank 20 (`content_8c19996aa890`)** | Score: 0 | Action: `monitor` | Reason: `below_threshold` | Confidence: False Negative (Severe)
    - *Why it's there:* 509k impressions, pos 2.5, actively declining, but updated 20 days ago.
    - *What would make it wrong:* Another critical false negative; page sits on top of Page 1 and is losing rank due to sudden competitor action or algorithmic updates that staleness rules cannot detect.

In [3]:
# Display Top-20 Queue Table for Skeptic Review
top20 = ranked_queue.head(20).copy()
display_cols = [
    'rank',
    'content_id',
    'client_id',
    'baseline_action_score',
    'action_label',
    'reason_code',
    'impressions_90d',
    'clicks_90d',
    'avg_position',
    'ctr',
    'days_since_last_update',
    'trend_direction',
    'is_declining_label'
]

print("=== TOP-20 SKEPTIC'S AUDIT QUEUE ===")
print(top20[display_cols].to_string(index=False, formatters={
    'baseline_action_score': '{:,.0f}'.format,
    'impressions_90d': '{:,}'.format,
    'clicks_90d': '{:,}'.format,
    'avg_position': '{:.1f}'.format,
    'ctr': '{:.2f}%'.format,
    'days_since_last_update': '{:.0f}'.format
}))

=== TOP-20 SKEPTIC'S AUDIT QUEUE ===
 rank           content_id         client_id baseline_action_score action_label        reason_code impressions_90d clicks_90d avg_position   ctr days_since_last_update trend_direction  is_declining_label
    1 content_cf56e2e2e282 client_7f2253d7e2                61,678      refresh stale_visible_page          61,678         94         19.7 0.15%                    194            down                   1
    2 content_7368877ea310 client_7f2253d7e2                59,472      refresh stale_visible_page          59,472         77         24.8 0.13%                    194            down                   1
    3 content_1bfaa38ff26c client_7f2253d7e2                25,715      refresh stale_visible_page          25,715         60         22.2 0.23%                    194            down                   1
    4 content_0a91db491d14 client_7f2253d7e2                13,299      refresh stale_visible_page          13,299         65         10.5 0.49%   

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Skeptic's Analysis of Weak Picks
Our top-20 review immediately surfaces three fundamental weaknesses of hand-crafted heuristics that our Week 5 model must solve:

1. **False Positives on Stale Stable Content (Rank 12):**
   `content_bdbec75c1148` has been untouched for 194 days and has 1,316 impressions, so the rule gave it a score of 1,316. But its observed trend is `stable` (`is_declining_label == 0`). Editorial bandwidth spent rewriting this page produces zero recovery value.

2. **Deep SERP Resource Waste (Ranks 5, 7, 17):**
   Pages ranking at positions 31.0 to 48.0 (Pages 4–5) with 0 to 1 click in 90 days. Because the rule multiplies impressions directly without position weighting, high-impression low-intent queries surface pages that lack the topical authority to ever reach Page 1 through text refreshes alone.

3. **Catastrophic False Negatives via Rigid Step Functions (Ranks 18, 20):**
   The single biggest failure mode: `content_5fe46e04994d` ranks at position 4.2 with 517,715 impressions and is actively collapsing (`trend_direction == 'down'`). Because it was updated 104 days ago (missing the arbitrary 180-day cutoff), the rule assigned it a score of 0. Hard thresholds create massive blind spots for flagship assets.

4. **Client Concentration Bias:**
   All of the top 10 flagged items come from a single client (`client_7f2253d7e2`). The rule is reflecting client-level batch update cadences rather than true portfolio-wide opportunity.

### Strict Leakage Audit
To guarantee validity, we verify that no future-window performance metrics or target proxies leaked into the rule inputs:
- Target columns strictly excluded: `trend_direction`, `trend_pct`, `is_declining_label`.
- Future-window columns excluded: `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`.
- Product outputs excluded: `health_score`, `priority_score`, `action_type`.
- **Rule Inputs Used:** Strictly observable trailing signals: `days_since_last_update` and `impressions_90d`.

In [4]:
# Programmatic Leakage Audit & Verification
FORBIDDEN_COLUMNS = [
    # Target proxies and labels
    'trend_direction', 'trend_pct', 'is_declining_label',
    # Temporal outcome / future window performance
    'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    # FlyRank internal product rule outputs
    'health_score', 'priority_score', 'action_type'
]

RULE_INPUT_COLUMNS = ['days_since_last_update', 'impressions_90d']

# Check for leakage overlap
leaked_features = [col for col in RULE_INPUT_COLUMNS if col in FORBIDDEN_COLUMNS]
assert len(leaked_features) == 0, f"CRITICAL LEAKAGE DETECTED: {leaked_features}"

print("=== LEAKAGE AUDIT REPORT ===")
print(f"Target-derived & future window columns checked: {len(FORBIDDEN_COLUMNS)}")
print(f"Rule input columns: {RULE_INPUT_COLUMNS}")
print("Leakage Overlap: NONE (0 forbidden columns found in rule formulation)")
print("Status: PASSED -- Baseline rule is 100% free of target leakage and temporal contamination.\n")

# Highlight audited weak picks
fp_row = top20[top20['rank'] == 12].iloc[0]
fn_row = top20[top20['rank'] == 18].iloc[0]
print("=== CONFIRMED WEAK PICKS FROM HAND REVIEW ===")
print(f"1. False Positive (Rank {fp_row['rank']}): {fp_row['content_id']} | Score: {fp_row['baseline_action_score']:.0f} | Action: {fp_row['action_label']} | Trend: {fp_row['trend_direction']} (STABLE - Wasted effort)")
print(f"2. False Negative (Rank {fn_row['rank']}): {fn_row['content_id']} | Score: {fn_row['baseline_action_score']:.0f} | Action: {fn_row['action_label']} | Trend: {fn_row['trend_direction']} ({fn_row['impressions_90d']:,} imp - Missed decay)")

=== LEAKAGE AUDIT REPORT ===
Target-derived & future window columns checked: 12
Rule input columns: ['days_since_last_update', 'impressions_90d']
Leakage Overlap: NONE (0 forbidden columns found in rule formulation)
Status: PASSED -- Baseline rule is 100% free of target leakage and temporal contamination.

=== CONFIRMED WEAK PICKS FROM HAND REVIEW ===
1. False Positive (Rank 12): content_bdbec75c1148 | Score: 1316 | Action: refresh | Trend: stable (STABLE - Wasted effort)
2. False Negative (Rank 18): content_5fe46e04994d | Score: 0 | Action: monitor | Trend: down (517,715 imp - Missed decay)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.